# 다이아몬드 — 회귀 완성본

원본: 카페 Untitled6 (2).ipynb.

흐름: 분포 → 이상치 → 인코딩 → 연습/시험 → 표준화 → 선형·규제 → 교차검증 → 로그.
`08_diamonds_eda.ipynb`는 EDA·전처리까지, 이 파일은 회귀까지 완성본입니다.


## 1. 도구 불러오기

표를 읽고 그림을 그리려면 숫자·표·그림 도구가 필요합니다.


In [1]:
# 숫자 표·그림 도구 불러오기
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [2]:
# seaborn 없으면 한 번만 설치 (이미 있으면 넘어감)
pip install seaborn


SyntaxError: invalid syntax (3859731936.py, line 2)

In [ ]:
# seaborn = matplotlib 위에 얹은 그림 도구 (히스토그램·박스·바이올린이 쉬움)
import seaborn as sns


## 2. 데이터 불러오기

**seaborn 내장 표** diamonds 를 씁니다.  
예전 수업 경로 C:/run/diamonds.csv 와 같은 다이아몬드 표입니다.


In [ ]:
# 내장 다이아몬드 표 불러오기 (절대경로 CSV 대신)
data = sns.load_dataset('diamonds')
data


## 3. 분포 보기 (EDA)

**EDA** = 모델 전에 그림·표로 데이터를 살펴보는 일입니다.  
**히스토그램** = 값이 어디에 몰렸는지. **박스플롯** = 가운데·수염·튀는 점.


In [ ]:
# 숫자 칸 분포를 2행 3열로 그리기
columns = ['carat', 'depth', 'table', 'x', 'y', 'z']
plt.figure(figsize=(10, 10))

for i, columns in enumerate(columns, 1):
    plt.subplot(2, 3, i)  # 2행 3열 / i번째 칸
    sns.histplot(data[columns], kde=True, bins=30)  # kde: 부드러운 밀도 곡선
plt.show()


In [ ]:
# 같은 칸을 박스플롯으로 — 튀는 점(이상치 후보)이 잘 보임
columns = ['carat', 'depth', 'table', 'x', 'y', 'z']
plt.figure(figsize=(10, 10))

for i, columns in enumerate(columns, 1):
    plt.subplot(2, 3, i)  # 2행 3열 / i번째 칸
    sns.boxplot(data[columns])
plt.show()


## 4. IQR로 이상치 줄이기

**이상치** = 다른 점과 너무 동떨어진 값입니다.  
가운데 50% 폭(**IQR**)의 1.5배 밖을 잘라 모델이 튀는 값에 덜 흔들리게 합니다.


In [ ]:
# 칸마다 IQR 밖 행을 차례로 걸러 냄
columns = ['carat', 'depth', 'table', 'x', 'y', 'z']

for col in columns:
    Q1 = data[col].quantile(0.25)  # 하위 25% 지점
    Q3 = data[col].quantile(0.75)  # 상위 25% 지점
    IQR = Q3 - Q1  # 가운데 50% 폭

    # IQR의 1.5배 밖 = 이상치 후보
    lower = Q1 - 1.5 * IQR  # 아래쪽 경계
    upper = Q3 + 1.5 * IQR  # 위쪽 경계

    # 경계 안쪽만 남김 (모든 숫자 칸에 대해 반복)
    data = data[(data[col] >= lower) & (data[col] <= upper)]


In [ ]:
# 이상치 제거 후 남은 표 확인
data


## 5. 컷 품질과 캐럿

**바이올린 플롯** = 값 분포의 두께를 보여 줍니다.  
컷이 좋을수록 캐럿이 작은 편이 보일 수 있습니다.


In [ ]:
# 컷 등급별 캐럿 분포
plt.figure(figsize=(10, 5))
sns.violinplot(x='cut', y='carat', data=data, order=['Fair', 'Good', 'Very Good', 'Premium', 'Ideal'])
plt.show()


In [ ]:
# 칸마다 비어 있는 값 개수 확인
data.isna().sum()


## 6. 글자 칸 → 숫자 (인코딩)

모델은 숫자만 받습니다.  
cut / color / clarity를 **등급 순서**로 번호를 매깁니다.


In [ ]:
# 글자 칸 → 숫자 (등급 순서 직접 지정)
from sklearn.preprocessing import LabelEncoder

cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order = ['D', 'E', 'F', 'G', 'H', 'I', 'J']
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

def ordered_label(series, order):
    la = LabelEncoder()
    la.fit(order)  # 등급 목록으로 맞춤
    return la.transform(series)

# 글자 칸을 숫자로
data['cut'] = ordered_label(data['cut'], cut_order)
data['color'] = ordered_label(data['color'], color_order)
data['clarity'] = ordered_label(data['clarity'], clarity_order)


In [ ]:
# CSV에만 있던 행 번호 칸이 있으면 제거 (seaborn 내장 표에는 보통 없음)
if 'Unnamed: 0' in data.columns:
    data.drop('Unnamed: 0', axis=1, inplace=True)


## 7. 힌트 칸 / 맞출 값 나누기

**피처(features)** = 맞출 때 쓰는 힌트 칸.  
**라벨(label)** = 맞출 값 price.


In [ ]:
# price=맞출 값, 나머지=힌트 칸
features = data.drop('price', axis=1)
label = data['price']


In [ ]:
# 학습에 쓰는 칸 이름 확인
x_col = features.columns
x_col


## 8. 연습·시험 나누기 & 표준화

연습으로 배우고, 시험으로 점수를 봅니다.
**표준화(StandardScaler)** = 칸마다 평균 0·분산 1 근처로 맞춤.
fit은 연습 표에만, 시험 표는 transform만 합니다.


In [ ]:
# 80% 연습 / 20% 시험 (random_state로 나눔을 고정)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(features, label, test_size=0.2, random_state=42)


In [ ]:
# 나눈 뒤 표준화 — 연습 표로 기준을 잡고, 시험 표는 그 기준으로만 변환
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
sc_x_train = sc.fit_transform(X_train)
sc_x_test = sc.transform(X_test)


## 9. 점수 함수 준비

**MSE** = 틀린 정도(제곱)의 평균 → 작을수록 좋음.  
**R²** = 설명력 → 1에 가까울수록 좋음.

아래 eval_data 에 모델을 넣어 연습/시험 점수를 같이 출력합니다.


In [ ]:
# 연습·시험 MSE / R² 한 번에 출력
from sklearn.metrics import mean_squared_error, r2_score

def eval_data(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    train_mse1 = mean_squared_error(y_train, y_pred_train)
    train_r2_score1 = r2_score(y_train, y_pred_train)

    test_mse1 = mean_squared_error(y_test, y_pred_test)
    test_r2_score1 = r2_score(y_test, y_pred_test)

    print(f'[Train] MSE:{train_mse1}, r2_score:{train_r2_score1}')
    print(f'[Test] MSE:{test_mse1}, r2_score:{test_r2_score1}')


## 10. 선형 회귀 & 규제 모델

**선형 회귀** = 직선(평면)으로 가격을 맞춤.  
**규제** = 가중치를 너무 키우지 못하게 눌러 **과적합**(연습만 잘 맞음)을 줄입니다.

- **Ridge**: 가중치 제곱을 눌러 전체적으로 작게
- **Lasso**: 절대값으로 눌러 덜 중요한 칸 가중치를 0에 가깝게
- **ElasticNet**: Ridge + Lasso 섞음


In [ ]:
# 기본 선형 회귀 (규제 없음, 표준화 전 X)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

lr = LinearRegression()  # 오차 제곱합만 최소화하는 기본 모델
eval_data(lr, X_train, X_test, y_train, y_test)
# 다음 셀부터 규제 모델


In [ ]:
# alpha가 클수록 규제 강함 → 가중치가 0에 가까워짐
# alpha가 작을수록 규제 약함 → 일반 선형 회귀에 가까움
ridge = Ridge(alpha=1.0)
eval_data(ridge, sc_x_train, sc_x_test, y_train, y_test)


In [ ]:
# Lasso: 덜 중요한 특성 가중치를 0으로 만들 수 있음
lasso = Lasso(alpha=1.0)
eval_data(lasso, sc_x_train, sc_x_test, y_train, y_test)


In [ ]:
# alpha: 규제 강도 / l1_ratio: 0이면 Ridge쪽, 1이면 Lasso쪽 (0.5 = 반반)
el = ElasticNet(alpha=0.1, l1_ratio=0.5)
eval_data(lasso, sc_x_train, sc_x_test, y_train, y_test)


## 11. 교차검증

**교차검증** = 표를 여러 겹으로 나눠 번갈아 시험해 평균 점수를 봅니다.  
한 번 나눈 시험 점수보다 덜 들쭉날쭉합니다.


In [ ]:
# 5겹 교차검증으로 R² 평균을 계산
from sklearn.model_selection import cross_val_score, KFold

def eval_cv(model, features, label, cv=5):
    kf = KFold(n_splits=cv, shuffle=True, random_state=42)

    # 각 겹에서 R²를 구한 뒤 평균
    r2_score = cross_val_score(model, features, label, cv=kf, scoring='r2')
    mean_r2_score = np.mean(r2_score)
    print(mean_r2_score)


In [ ]:
# 네 모델을 같은 방식으로 교차검증 (원본과 같이 시험 표로 호출)
eval_cv(lr, X_test, y_test, cv=5)
eval_cv(ridge, X_test, y_test, cv=5)
eval_cv(lasso, X_test, y_test, cv=5)
eval_cv(el, X_test, y_test, cv=5)


## 12. 가격 분포와 로그 변환

가격이 싼 쪽에 많이 몰리면(**왜도**), 큰 값이 학습을 흔들 수 있습니다.  
log1p = log(1+x) 로 눌러 분포를 고르게 만듭니다.


In [ ]:
# 가격 히스토그램 (싼 쪽이 많음)
plt.hist(label)
# 치우침(왜도) → 다음 셀에서 로그


In [ ]:
# log1p 연습 메모 (실행해도 그림은 안 그림)
# x=np.exp(3) #20.xxx
# y=np.log1p(x) #log(1+x) log(1+20.xxx) = log(21.xxx) =3.xxx
# print(y)

# #np.log(0) ->음의 무한대
# #np.log1p(0) -> log(1) =0


In [ ]:
# price에 로그 변환을 적용한 뒤 히스토그램 출력
eq = np.log1p(label)  # log(1+price)
plt.hist(eq)
